In [ ]:
# pip install xlsxwriter
# pip install requests
# pip install beautifulsoup4
# pip install pandas


import requests
from bs4 import BeautifulSoup
import pandas as pd
url = "https://search.naver.com/search.naver"
my_params = {"where":"news","query":"인공지능"}
response = requests.get(url,params=my_params,timeout=10)
# print("상태코드",response.status_code)
# print("내용",response.text)
soup = BeautifulSoup(response.text, "html.parser")
# print(soup.prettify())
news_list = soup.select(".fds-news-item-list-tab > div")
# print(len(news_list))
data=[]
for item in news_list:
    title = item.select_one('a[data-heatmap-target=".tit"]')
    desc = item.select_one('a[data-heatmap-target=".body"]')
    press = item.select_one(".sds-comps-profile-info-title")
    url = title.get("href")
    # print(title.get_text(" ",strip=True).replace("새 창 열림","").strip())
    # print(desc.get_text(" ",strip=True).strip())
    # print(press.get_text(" ",strip=True).strip().replace("새 창 열림","").strip())
        # print("==="*30)
    strip_title =  title.get_text(" ",strip=True).replace("새 창 열림","").strip()
    strip_press= press.get_text(" ",strip=True).strip().replace("새 창 열림","").strip()
    strip_desc = desc.get_text(" ",strip=True).strip()


    data.append({"제목":strip_title,"언론사":strip_press,"내용":strip_desc,"URL":url})
df = pd.DataFrame(data)
df.to_excel("네이버뉴스.xlsx",index=False,engine="xlsxwriter")




        


In [ ]:
# pip install xlsxwriter
# pip install requests
# pip install beautifulsoup4
# pip install pandas


import requests
from bs4 import BeautifulSoup
import pandas as pd
url = "https://search.daum.net/search"
my_params = {"w":"news","q":"인공지능"}
response = requests.get(url,params=my_params,timeout=10)
# print("상태코드",response.status_code)
# print("내용",response.text)
soup = BeautifulSoup(response.text, "html.parser")
# print(soup.prettify())
news_list = soup.select(".c-list-basic > li")
# print(len(news_list))
data=[]
for item in news_list:
    title = item.select_one(".item-title a")
    strip_title=title.get_text(" ",strip=True)
    desc = item.select_one(".item-contents")
    strip_desc= desc.get_text(" ",strip=True)
    press= item.select_one(".c-tit-doc .txt_info")
    strip_press= press.get_text(" ",strip=True)

    url = title.get("href")

    data.append({"제목":strip_title,"언론사":strip_press,"내용":desc,"URL":url})
df = pd.DataFrame(data)
df.to_excel("다음뉴스.xlsx",index=False,engine="xlsxwriter")




        


In [39]:
# pip install xlsxwriter
# pip install requests
# pip install beautifulsoup4
# pip install pandas


import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
url = "https://search.daum.net/search"
data=[]
for page in range(1,11):
    print(page)
    my_params = {"w":"news","q":"인공지능","p":page}
    response = requests.get(url,params=my_params,timeout=10)
    # print("상태코드",response.status_code)
    # print("내용",response.text)
    soup = BeautifulSoup(response.text, "html.parser")
    # print(soup.prettify())
    news_list = soup.select(".c-list-basic > li")
    # print(len(news_list))
    
    for item in news_list:
        title = item.select_one(".item-title a")
        strip_title=title.get_text(" ",strip=True)
        desc = item.select_one(".item-contents")
        strip_desc= desc.get_text(" ",strip=True)
        press= item.select_one(".c-tit-doc .txt_info")
        strip_press= press.get_text(" ",strip=True)

        news_url = title.get("href")

        data.append({"제목":strip_title,"언론사":strip_press,"내용":strip_desc,"URL":news_url})
    time.sleep(1)
df = pd.DataFrame(data)
df.to_excel("다음뉴스.xlsx",index=False,engine="xlsxwriter")




        


1
2
3
4
5
6
7
8
9
10


In [41]:
# pip install selenium
from selenium import webdriver
driver = webdriver.Chrome()



In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

driver = webdriver.Chrome()
driver.get("https://search.naver.com/search.naver")

# 검색
search = driver.find_element(By.NAME, "query")
search.send_keys("트럼프 중간선거")
search.send_keys(Keys.ENTER)

wait = WebDriverWait(driver, 10)

# 뉴스 더보기 클릭
more = wait.until(
    EC.element_to_be_clickable(
        (By.CSS_SELECTOR, 'a[data-heatmap-target=".more"]')
    )
)
more.click()

# 뉴스 목록 로딩 대기
wait.until(
    EC.presence_of_element_located(
        (By.CSS_SELECTOR, ".fds-news-item-list-tab > div")
    )
)

# 스크롤 10회
for i in range(10):
    driver.execute_script(
        "window.scrollBy(0, 800);"
    )
    time.sleep(2)

# 뉴스 목록 수집
news_list = driver.find_elements(
    By.CSS_SELECTOR, ".fds-news-item-list-tab > div"
)

print("뉴스 개수:", len(news_list))

data = []

for news in news_list:

    titles = news.find_elements(
        By.CSS_SELECTOR,
        'a[data-heatmap-target=".tit"]'
    )

    descs = news.find_elements(
        By.CSS_SELECTOR,
        'a[data-heatmap-target=".body"]'
    )

    presses = news.find_elements(
        By.CSS_SELECTOR,
        ".sds-comps-profile-info-title-text"
    )

    if not titles:
        continue

    title = titles[0].text
    link = titles[0].get_attribute("href")
    desc = descs[0].text if descs else ""
    press = presses[0].text if presses else ""

    data.append({
        "제목": title,
        "내용": desc,
        "언론사": press,
        "링크": link
    })

# 엑셀 저장
df = pd.DataFrame(data)
df.to_excel("네이버뉴스.xlsx", index=False)

print("수집 완료:", len(df), "개")

driver.quit()

뉴스 개수: 36
수집 완료: 36 개
